# Ablation Study

This notebook runs the ablation evaluation from `backend/evaluation/evaluate_ablation.py` and writes the results to `evaluation/results/ablation_results.json`.

In [5]:
from pathlib import Path
import json
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'backend').exists():
            return candidate
    raise RuntimeError('Could not locate repository root')

repo_root = find_repo_root(Path.cwd())
backend_dir = repo_root / 'backend'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

# Import the ablation module constants
from backend.evaluation.evaluate_ablation import RESULTS_OUTPUT

print(f"Results will be saved to: {RESULTS_OUTPUT}")
print(f"Repository root: {repo_root}")

/home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration loaded:
  - API Host: 0.0.0.0:8000
  - Upload Directory: uploads
  - Max File Size: 50 MB
  - Groq API Configured: True
  - Qdrant Configured: True
  - Qdrant Collection: research_papers
  - LangSmith Tracing: Enabled
Results will be saved to: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results/ablation_results.json
Repository root: /home/aman/storage/Python/Projects/Research Paper Assistant


In [6]:
from pathlib import Path
import json

results_path = Path(RESULTS_OUTPUT)
if results_path.exists():
    data = json.loads(results_path.read_text(encoding='utf-8'))
    summary = data.get('summary', {})
    print(f'Results loaded from {results_path}')
    for item in summary.get('configurations', []):
        print(
            f"{item['name']}: P@3={item['precision_at_3']}, P@5={item['precision_at_5']}, "
            f"R@5={item['recall_at_5']}, MRR={item['reciprocal_rank']}"
        )
    improvement = summary.get('improvement', {})
    print(
        f"Improvement: +{improvement.get('precision_at_5', 0)} P@5, "
        f"+{improvement.get('reciprocal_rank', 0)} MRR"
    )
else:
    print(f'No saved results found yet at {results_path}.')
    print('Run backend/evaluation/evaluate_ablation.py in a terminal to generate them.')

Results loaded from /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/results/ablation_results.json
Dense Only: P@3=0.12, P@5=0.08, R@5=0.22, MRR=0.18
Dense + BM25 (no reranker): P@3=0.12, P@5=0.08, R@5=0.23, MRR=0.19
Full System: P@3=0.16, P@5=0.09, R@5=0.27, MRR=0.28
Improvement: +0.01 P@5, +0.1 MRR


In [7]:
from pathlib import Path
import json
import pandas as pd

results_path = Path(RESULTS_OUTPUT)
if results_path.exists():
    data = json.loads(results_path.read_text(encoding='utf-8'))
    summary = data.get('summary', {})
    
    # Create a results table
    configs = summary.get('configurations', [])
    results_df = pd.DataFrame([
        {
            'Configuration': c['name'],
            'P@3': c['precision_at_3'],
            'P@5': c['precision_at_5'],
            'R@5': c['recall_at_5'],
            'MRR': c['reciprocal_rank'],
            'Questions': c['num_questions']
        }
        for c in configs
    ])
    
    print("Ablation Study Results:")
    print("=" * 80)
    print(results_df.to_string(index=False))
    
    improvement = summary.get('improvement', {})
    print("\n" + "=" * 80)
    print(f"Improvement (Dense Only → Full System):")
    print(f"  Precision@5: +{improvement.get('precision_at_5', 0)}")
    print(f"  Mean Reciprocal Rank: +{improvement.get('reciprocal_rank', 0)}")
    print("=" * 80)
else:
    print(f'No saved results found at {results_path}.')
    print('Run backend/evaluation/evaluate_ablation.py in a terminal to generate them.')

Ablation Study Results:
             Configuration  P@3  P@5  R@5  MRR  Questions
                Dense Only 0.12 0.08 0.22 0.18         32
Dense + BM25 (no reranker) 0.12 0.08 0.23 0.19         32
               Full System 0.16 0.09 0.27 0.28         32

Improvement (Dense Only → Full System):
  Precision@5: +0.01
  Mean Reciprocal Rank: +0.1
